# Langchain Agents 
This notebook demonstrates how to create and use agents in Langchain, including the use of tools like DuckDuckGoSearch for real-time information retrieval.

At the end of this notebook, we can see how to create a ReAct agent from a prompt template and use it with a memory system to maintain chat history.



In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq


In [2]:
# load environment variables from .env file
load_dotenv()

True

In [3]:
# Get llms models via the `langchain_groq` package
llm_model = ChatGroq(
    model="llama-3.3-70b-versatile", #llama-3.3-70b-versatile",
    temperature=.5,
    max_tokens=512,
)

## Langchain Tools

The [third-party tools](https://python.langchain.com/docs/integrations/tools/) from the langchain-community ecosytem allow us to extent the capabilities of our agents

In [4]:
# from langchain_community.tools import DuckDuckGoSearchRun
from ddgs import DDGS
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

`DuckDuckGoSearchRun` allows us to perform web searches and retrieve information in real-time, which can be very useful for agents that need to answer questions based on current data.


In [5]:
#api_wrapper = DuckDuckGoSearchAPIWrapper(region="en-en",max_results=10)
#search = DuckDuckGoSearchRun()#,api_wrapper=api_wrapper)

def ddg_search(query: str):
    with DDGS() as ddgs:
        return ddgs.text(query, max_results=5)
search = ddg_search

In [6]:
results = search("Prime Italian Minister")  

In [7]:
results

[{'title': 'Giorgia Meloni - Wikipedia',
  'href': 'https://en.wikipedia.org/wiki/Giorgia_Meloni',
  'body': 'Giorgia Meloni ( Italian : [ˈdʒordʒa meˈloːni]; born 15 January 1977) is an Italian politician who has served as Prime Minister of Italy since October 2022. She is the first woman to hold the office and the head of the third-longest government in the history of the Italian Republic.'},
 {'title': 'List of prime ministers of Italy - Wikipedia',
  'href': 'https://en.wikipedia.org/wiki/List_of_prime_ministers_of_Italy',
  'body': 'List of prime ministers of Italy Top left: Camillo Benso, Count of Cavour was the first prime minister of the Kingdom of Italy. Top right: Benito Mussolini was the longest-serving prime minister in Italian history. Bottom left: Silvio Berlusconi was the longest-serving prime minister of the Italian Republic.'},
 {'title': 'List Of Prime Minister Of Italy (2025 Updated) - GkGigs',
  'href': 'https://gkgigs.com/prime-minister-italy/',
  'body': 'The Prime

In [8]:
# Define a tool for the DuckDuckGo search
from langchain.tools import Tool
search_tool = Tool(
    name="DuckDuckGoSearch",
    func=ddg_search,
    description="Use this tool when you need to search for real-time information from DuckDuckGo."
)

In [9]:
from langchain.agents import initialize_agent, AgentType


In [10]:
# Initialize the agent with the DuckDuckGo search tool
agent = initialize_agent(
    tools=[search_tool],
    llm=llm_model,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    max_iterations=7,
    agent_name="DuckDuckGoSearchAgent",
    agent_description="An agent that can search the web using DuckDuckGoSearch and summarize findings clearly."
)

# Now we can use the agent to answer questions
results = agent.invoke("Who is the Italian Prime Minister?")  # Example query to the agent

d:\Programming\ai_ds_bootcamp\ds-ai-agent\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 0.3.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  warn_deprecated(




> Entering new AgentExecutor chain...
To find out who the current Italian Prime Minister is, I should search for the most recent information available.

Action: DuckDuckGoSearch
Action Input: Current Italian Prime Minister

(I'll wait for the search results...)
Observation: [{'title': 'Giorgia Meloni - Wikipedia', 'href': 'https://en.wikipedia.org/wiki/Giorgia_Meloni', 'body': 'Prime Minister of Italy since 2022. Giorgia Meloni. Official portrait, 2024.On 18 October 2024, Meloni traveled to Lebanon and assured the solidarity of Italy with Lebanon and UNIFIL in the current conflict.'}, {'title': 'Wikipedia', 'href': 'https://www.wikipedia.org/', 'body': 'Afrikaans Shqip العربية Asturianu Azərbaycanca Български 閩南語 / Bân-lâm-gú বাংলা Беларуская Català Čeština Cymraeg Dansk Deutsch Eesti Ελληνικά English Español Esperanto Euskara فارسی Français Galego 한국어 Հայերեն हिन्दी Hrvatski Bahasa Indonesia Italiano עברית...'}, {'title': 'www.governo.it | Governo Italiano Presidenza del Consiglio d

### Create a an Agent with DuckDuckGoSearch with Memory
In this section, we will create an agent that can search the web using DuckDuckGoSearch and maintain a memory of previous interactions. 

In [11]:
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder


In [12]:

# Build a chat prompt that includes conversation history so the agent can resolve references.
prompt_template = ChatPromptTemplate.from_messages([
    ("system", """Answer the following questions as best you can. You have access to the following tools:
{tools}
Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    ("assistant", "{agent_scratchpad}"),
])


The react_template instructs the laguage model to:
1. __Think__ about how to solve a user's question
2. __Use tools__ (actions) to gather mor information or perform actions
3. __Observe results__ and update the its thoughts
4. Eventually, reach a __Final Answer__

The following is the explanation of the `react_template` used to create the agent:

```python
react_template = """
'Answer the following questions as best you can. You have access to the following tools:

{tools}
```
+ Introduce the agent and tells it what tools it can use
+ {tools} is a placeholder for the tools that will be used by the agent

```text
Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question
```

This is the strict __format__ the LLM must follow to:

* Show its reasoning (Thought)

* Choose and execute a tool (Action)

* Feed it input (Action Input)

* Observe results (Observation)

* Loop until it gets to the final answer

```text
Begin!
Question: {input}
Thought:{agent_scratchpad}'
"""
```
+ `{input}` is the actual user question.

+ `{agent_scratchpad}` holds any intermediate steps already completed (e.g., previous Thoughts, Actions, Observations).

+ The agent continues from this point.

In [ ]:
# prompt_template defined above with chat history support


In [13]:
from langchain.agents import AgentExecutor, create_react_agent
from langchain.memory import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [14]:
# Simple session store to keep track of chat history per conversation
store = {}


def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


In [15]:
# Create the agent 
agent = create_react_agent(llm_model, [search_tool], prompt_template)
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool],
    verbose=True,
    handle_parsing_errors=True,
)


In [16]:
# Wrap the agent executor with message history
agent_with_chat_history = RunnableWithMessageHistory(
    agent_executor,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)


In [17]:
# Now we can use the agent with chat history to answer questions
agent_with_chat_history.invoke(
    {"input": "How many people live in italy?"},
    config={"configurable": {"session_id": "1"}},
)

Parent run 8a9141d4-f92a-4f74-a11a-f6172070ac25 not found for run 4cab7cf6-5b36-4869-b50b-188b091e6d87. Treating as a root run.




> Entering new AgentExecutor chain...
Thought: To find the current population of Italy, I need to search for the most recent data available.

Action: DuckDuckGoSearch
Action Input: Italy population 2024
[{'title': 'Demographics of Italy - Wikipedia', 'href': 'https://en.wikipedia.org/wiki/Demographics_of_Italy', 'body': 'Animated population pyramid 1982-2021. Those born during the World wars are marked in dark The demography of Italy is monitored by the Italian National Institute of Statistics (Istat). At the beginning of 2024 , Italy had an estimated population of 58.9 million. [1] Its population density, at 195.7 inhabitants per square kilometre (507/sq mi), is higher than both the EU (106.6/km2) and ...'}, {'title': 'Italy Population (2025) - Worldometer', 'href': 'https://www.worldometers.info/world-population/italy-population/', 'body': 'Population of Italy : current, historical, and projected population , growth rate, immigration, median age, total fertility rate (TFR), populat

{'input': 'How many people live in italy?',
 'chat_history': [],
 'output': 'The population of Italy is approximately 58.9 million.'}

In [18]:
# Let's ask another question to see how the agent uses the chat history
agent_with_chat_history.invoke(
    {"input": "what is their national anthem called?"},
    config={"configurable": {"session_id": "1"}},
)

Parent run 336339fa-f618-4f70-8e46-df20e5ef2f92 not found for run 6f3a0cae-5453-4dc0-93df-4c4ffe18849d. Treating as a root run.




> Entering new AgentExecutor chain...
Question: what is the national anthem of Italy called?
Thought: I should search for the name of Italy's national anthem
Action: DuckDuckGoSearch
Action Input: Italy national anthem[{'title': 'Il Canto degli Italiani - Wikipedia', 'href': 'https://en.wikipedia.org/wiki/Il_Canto_degli_Italiani', 'body': '2 weeks ago - "Il Canto degli Italiani", is a patriotic song written by Goffredo Mameli and set to music by Michele Novaro in 1847, currently used as the national anthem of Italy. It is known among Italians as the "Inno di Mameli"—after the author of the lyrics, or "Fratelli d\'Italia"—from the song\'s ...'}, {'title': 'Italy National Anthem', 'href': 'https://georgewbush-whitehouse.archives.gov/national-anthem/rome-full.html', 'body': "Italy National Anthem: Inno di Mameli (Hymn of Mameli) Listen to the National Anthem · Original Italian Words · Fratelli d'Italia L'Italia s'è desta Dell'elmo di Scipio S'è cinta la testa · Dov'è la vittoria? Le por

{'input': 'what is their national anthem called?',
 'chat_history': [HumanMessage(content='How many people live in italy?'),
  AIMessage(content='The population of Italy is approximately 58.9 million.')],
 'output': 'The national anthem of Italy is called "Il Canto degli Italiani", also known as "Inno di Mameli" or "Fratelli d\'Italia"'}

### Exercise
Create or Select some tools from the [langchain-community tools](https://python.langchain.com/docs/integrations/tools/) and create a ReAct agent that can use them to solve certain tasks.

In [ ]:
from langchain_community.tools import YouTubeSearchTool

youtube_tool = YouTubeSearchTool(
    max_results=5,
    description="Use this tool to find YouTube videos when the user asks for tutorials, walkthroughs, or video explanations."
)

# Create the agent 
agent = create_react_agent(llm_model, [search_tool, youtube_tool], prompt_template)
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool, youtube_tool],
    verbose=True,
    handle_parsing_errors=True,
)

# Wrap the agent executor with message history
agent_with_chat_history = RunnableWithMessageHistory(
    agent_executor,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

# Now we can use the agent with chat history to answer questions
agent_with_chat_history.invoke(
    {"input": "Do they like dogs there?"},
    config={"configurable": {"session_id": "1"}},
)

Parent run 983efc53-4672-4a46-8df4-9fadd78c7bd2 not found for run 2860ce73-ddd7-4776-a897-352c13f66998. Treating as a root run.




> Entering new AgentExecutor chain...
The capital of Great Britain is London.Invalid Format: Missing 'Action:' after 'Thought: I now know the final answer 
Final Answer: London

> Finished chain.


{'input': 'What is the capital of Great Britain?',
 'chat_history': [HumanMessage(content='How many people live in italy?'),
  AIMessage(content='The population of Italy is approximately 58.9 million.'),
  HumanMessage(content='what is their national anthem called?'),
  AIMessage(content='The national anthem of Italy is called "Il Canto degli Italiani", also known as "Inno di Mameli" or "Fratelli d\'Italia"'),
  HumanMessage(content='Find a YouTube channel PartORG that has many videos about gaming'),
  AIMessage(content='You can try searching on youtube with the keywords "PartORG gaming" or "PartORG gaming channel" or "PartORG gaming youtube channel" to find the channel.'),
  HumanMessage(content='Find a YouTube channel about Zwergschnauzers that travel the world'),
  AIMessage(content='Unfortunately, I was unable to find a specific YouTube channel about Zwergschnauzers that travel the world.'),
  HumanMessage(content='Find a channel about Zwergschnauzers that travel the world'),
  AIM

In [25]:
# Now we can use the agent with chat history to answer questions
agent_with_chat_history.invoke(
    {"input": "Do they like dogs there?"},
    config={"configurable": {"session_id": "1"}},
)

Parent run e3ecbd21-2db6-490f-91e0-75c1a04afa83 not found for run 45fc3e33-8f5c-48a9-987a-888968d05c6d. Treating as a root run.




> Entering new AgentExecutor chain...
Yes, the British are known to be dog lovers. Many people in the UK have dogs as pets, and dogs are often welcome in public places such as parks, pubs, and restaurants. In fact, the UK is home to many dog-friendly destinations and events, and dogs are often considered to be part of the family.Invalid Format: Missing 'Action:' after 'Thought: I will answer the question directly.
 Action: None Invalid Format: Missing 'Action Input:' after 'Action:' I now know the final answer
Final Answer: Yes, the British are known to be dog lovers and have a strong affection for dogs.

> Finished chain.


{'input': 'Do they like dogs there?',
 'chat_history': [HumanMessage(content='How many people live in italy?'),
  AIMessage(content='The population of Italy is approximately 58.9 million.'),
  HumanMessage(content='what is their national anthem called?'),
  AIMessage(content='The national anthem of Italy is called "Il Canto degli Italiani", also known as "Inno di Mameli" or "Fratelli d\'Italia"'),
  HumanMessage(content='Find a YouTube channel PartORG that has many videos about gaming'),
  AIMessage(content='You can try searching on youtube with the keywords "PartORG gaming" or "PartORG gaming channel" or "PartORG gaming youtube channel" to find the channel.'),
  HumanMessage(content='Find a YouTube channel about Zwergschnauzers that travel the world'),
  AIMessage(content='Unfortunately, I was unable to find a specific YouTube channel about Zwergschnauzers that travel the world.'),
  HumanMessage(content='Find a channel about Zwergschnauzers that travel the world'),
  AIMessage(conten